# CRS from the Scopus Author and Index Keywords

Not reported in the manuscript. The same constructor, tau = 0.40, w >= 20 and Louvain seed applied to four keyword sets of the same 52,946 documents: `llm_k5` (published keywords; gate: reproduces the published CRS), `scopus_all` (every author and index keyword of the record), `scopus_first5` (the first five, fixed cardinality) and `scopus_all_coword` (no semantic constraint). Arms are compared on vocabulary, fragmentation, connectivity, backbone size, modularity, hub share, English residue, backbone overlap and partition agreement.

Inputs: `EID_KEYWORDS.xlsx`, `data/insumo_row_to_eid.csv`, private record file from `FTTS_PRIVATE_DIR`. Outputs in `aditional_experiments/results/e10_scopus_keyword_baseline/`.

In [1]:
# ============================================================
# CONFIGURATION AND IMPORTS
# ============================================================
import os
os.environ["OMP_NUM_THREADS"] = "4"      # several notebooks may run concurrently on one machine

import contextlib, io, itertools, json, re, sys
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
torch.set_num_threads(4)

# The notebook lives in aditional_experiments/ but every path below is relative to the repository
# root; nbconvert and Jupyter start the kernel in the notebook's own folder, so move up one level.
if Path.cwd().name == "aditional_experiments":
    os.chdir(Path.cwd().parent)
print("working directory:", Path.cwd())

sys.path.insert(0, "scripts")
import common as C
from crs_reference import build_backbone, build_crs_for_tau

KEYWORDS = "EID_KEYWORDS.xlsx"
ALIGNMENT = "data/insumo_row_to_eid.csv"
INSUMO = C.PRIVATE_DIR / "corpus_insumo_DEFINITIVO.csv"      # private Scopus records (Elsevier licence)
OUT = Path("aditional_experiments/results/e10_scopus_keyword_baseline")
OUT.mkdir(parents=True, exist_ok=True)

LOUVAIN_SEEDS = list(range(42, 52))
SPANISH_STOP = {"de", "la", "el", "en", "y", "para", "del", "las", "los", "con", "una", "un", "por"}
THREADS = 4
_nonalnum = re.compile(r"[^a-z0-9 ]+")
_ws = re.compile(r"\s+")

if not INSUMO.exists():
    raise FileNotFoundError(
        f"private record file not found: {INSUMO}. Set FTTS_PRIVATE_DIR to the directory that holds "
        "corpus_insumo_DEFINITIVO.csv (Scopus records, not redistributed with this repository).")

# Results archived in the repository before this run, kept in memory for the final comparison cell.
archived = {}
for name in ("summary.json", "validation.json"):
    if (OUT / name).exists():
        archived[name] = json.load(open(OUT / name, encoding="utf-8"))
if (OUT / "arms_comparison.csv").exists():
    archived["arms_comparison.csv"] = pd.read_csv(OUT / "arms_comparison.csv")
print("archived files available for the final check:", sorted(archived))

C.set_seeds()
T = C.Timer()
meta = C.env_metadata(experiment="E10 Scopus keyword baseline", llm_calls=0, paid_api_calls=0,
                      inputs={"insumo": {"path": str(INSUMO), "sha256": C.sha256(INSUMO), "redistributed": False},
                              "keywords": {"path": KEYWORDS, "sha256": C.sha256(KEYWORDS)},
                              "alignment": {"path": ALIGNMENT, "sha256": C.sha256(ALIGNMENT)}})
print(f"tau = {C.TAU_EDGE} | backbone w >= {C.W_BACKBONE} | Louvain seed {C.SEED}")
for k, v in meta["inputs"].items():
    print(f"  {k:10s} sha256={v['sha256']}")

working directory: /home/mat/academic-writing/papers/from_text_to_structure/data_repo


archived files available for the final check: ['arms_comparison.csv', 'summary.json', 'validation.json']


/home/mat/academic-writing/papers/from_text_to_structure/data_repo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tau = 0.4 | backbone w >= 20 | Louvain seed 42
  insumo     sha256=597d4d8d44ddea693d4386798ea0a9d3380d1314782e937843e3184d78e2f274
  keywords   sha256=a440b7cedabea950ddb0be27610e1a58f747191ae08999801d73994f9d801c8c
  alignment  sha256=9d592a93d35cde582eb914e40334746f14fd13e743d56176cb8668de83707a03


In [2]:
# ============================================================
# HELPERS (vocabulary statistics, arm construction, document assignment)
# ============================================================
def notebook_clean(items):
    """Notebook 2 semantics: non-empty strings, strip().lower(), sorted(set())."""
    return sorted(set(str(k).strip().lower() for k in items if isinstance(k, str) and k.strip()))


def canon(phrase: str) -> str:
    """Aggressive canonical form used only to measure lexical fragmentation: strip punctuation and
    hyphens, collapse spaces, naive singular."""
    t = _ws.sub(" ", _nonalnum.sub(" ", phrase.lower())).strip()
    words = [w[:-1] if len(w) > 3 and w.endswith("s") and not w.endswith("ss") else w for w in t.split()]
    return " ".join(words)


def fragmentation(vocab):
    groups = defaultdict(list)
    for v in vocab:
        groups[canon(v)].append(v)
    multi = {k: sorted(v) for k, v in groups.items() if len(v) > 1}
    return {"vocabulary": len(vocab), "canonical_forms": len(groups),
            "share_vocabulary_that_collapses": 1 - len(groups) / max(1, len(vocab)),
            "variant_groups": len(multi),
            "largest_groups": sorted(multi.values(), key=len, reverse=True)[:15]}


def english_residue(flat):
    non_ascii = sum(any(ord(ch) > 127 for ch in k) for k in flat)
    spanish = sum(any(w in SPANISH_STOP for w in k.split()) for k in flat)
    return {"keywords": len(flat), "share_non_ascii": non_ascii / max(1, len(flat)),
            "share_spanish_function_words": spanish / max(1, len(flat))}


def hub_stats(H):
    if H.number_of_edges() == 0:
        return {}
    hub = max(H.degree(), key=lambda t: t[1])[0]
    return {"hub": hub, "hub_degree": int(H.degree(hub)), "hub_share_of_edges": H.degree(hub) / H.number_of_edges()}


def build_arm(name, docs, emb, tau, T):
    """Global graph, w >= 20 backbone and Louvain over ten seeds for one keyword source."""
    G = build_crs_for_tau(docs, emb, tau)
    T.mark(f"build_{name}")
    H = build_backbone(G, C.W_BACKBONE)
    parts, qs, ncs = {}, [], []
    for sd in LOUVAIN_SEEDS:
        p, q = C.louvain_partition(H, sd)
        parts[sd] = p; qs.append(q); ncs.append(len(set(p.values())) if p else 0)
    s = {"arm": name, "tau": tau, "documents": len(docs),
         "keywords_total": int(sum(len(d) for d in docs)),
         "keywords_per_document_mean": float(np.mean([len(d) for d in docs])),
         "keywords_per_document_median": float(np.median([len(d) for d in docs])),
         **{f"global_{k}": v for k, v in C.graph_summary(G).items()},
         **{f"backbone_{k}": v for k, v in C.graph_summary(H).items()},
         "backbone_modularity_seed42": qs[0],
         "backbone_modularity_mean": float(np.nanmean(qs)), "backbone_modularity_sd": float(np.nanstd(qs, ddof=1)),
         "backbone_communities_seed42": ncs[0], "backbone_communities_mean": float(np.mean(ncs)),
         **{f"backbone_{k}": v for k, v in hub_stats(H).items()}}
    # documents covered by the backbone (at least one keyword among backbone concepts)
    bb = set(H.nodes())
    s["share_documents_with_backbone_concept"] = float(np.mean([any(k in bb for k in d) for d in docs]))
    T.mark(f"backbone_{name}")
    return s, G, H, parts[LOUVAIN_SEEDS[0]]


def document_assignments(docs, part):
    """Majority community of a document's backbone keywords; None if no backbone keyword, 'tie' on ties."""
    out = []
    for d in docs:
        cnt = Counter(part[k] for k in d if k in part)
        if not cnt:
            out.append(None); continue
        top = cnt.most_common()
        out.append("tie" if len(top) > 1 and top[0][1] == top[1][1] else top[0][0])
    return out


print("helpers defined")

helpers defined


In [3]:
# ============================================================
# DOCUMENTS AND KEYWORD SETS (LLM versus Scopus author/index keywords)
# ============================================================
ins = pd.read_csv(INSUMO)["insumo"].astype(str).tolist()
align = C.load_alignment(Path(ALIGNMENT))
pub = C.load_published_keywords(Path(KEYWORDS), notebook_semantics=True)
eids, llm_docs, sc_all, sc_first5 = [], [], [], []
n_no_scopus = 0
for eid, row in zip(align.eid, align.insumo_row):
    kws = pub.get(eid, [])
    if not kws:                      # notebook 2 drops empty lists: 52,946 analysed documents
        continue
    rec = C.parse_record(ins[row])
    raw = [k for k in rec["original_keywords_raw"].split(";") if k.strip()]
    ordered = []                     # record order, de-duplicated after normalisation
    for k in raw:
        t = str(k).strip().lower()
        if t and t not in ordered:
            ordered.append(t)
    if not ordered:
        n_no_scopus += 1
    eids.append(eid); llm_docs.append(kws)
    sc_all.append(sorted(ordered)); sc_first5.append(sorted(ordered[:5]))
n = len(eids)
print(f"documents={n} (records without Scopus keywords: {n_no_scopus})")
T.mark("load")

vocab_llm = sorted(set(itertools.chain.from_iterable(llm_docs)))
vocab_sc = sorted(set(itertools.chain.from_iterable(sc_all)))
vocab_sc5 = sorted(set(itertools.chain.from_iterable(sc_first5)))
print(f"vocabulary: llm={len(vocab_llm)} scopus_all={len(vocab_sc)} scopus_first5={len(vocab_sc5)}")
frag = {"llm_k5": fragmentation(vocab_llm), "scopus_all": fragmentation(vocab_sc), "scopus_first5": fragmentation(vocab_sc5)}
residue = {"llm_k5": english_residue(list(itertools.chain.from_iterable(llm_docs))),
           "scopus_all": english_residue(list(itertools.chain.from_iterable(sc_all)))}
T.mark("vocabulary")

print("\nLexical fragmentation (aggressive canonical form):")
print(pd.DataFrame([{"arm": k, **{kk: vv for kk, vv in v.items() if kk != "largest_groups"}} for k, v in frag.items()])
      .round(4).to_string(index=False))
print("\nEnglish residue:")
print(pd.DataFrame([{"arm": k, **v} for k, v in residue.items()]).round(5).to_string(index=False))
print("\nLargest variant groups, LLM vocabulary:", frag["llm_k5"]["largest_groups"][:3])
print("Largest variant groups, Scopus vocabulary:", frag["scopus_all"]["largest_groups"][1:3])

documents=52946 (records without Scopus keywords: 0)


vocabulary: llm=56635 scopus_all=97738 scopus_first5=65420



Lexical fragmentation (aggressive canonical form):
          arm  vocabulary  canonical_forms  share_vocabulary_that_collapses  variant_groups
       llm_k5       56635            52404                           0.0747            3944
   scopus_all       97738            88301                           0.0966            8545
scopus_first5       65420            60632                           0.0732            4308

English residue:
       arm  keywords  share_non_ascii  share_spanish_function_words
    llm_k5    264586          0.00014                       0.00006
scopus_all    530670          0.00338                       0.00031

Largest variant groups, LLM vocabulary: [['student attitude', 'student attitudes', 'students attitude', 'students attitudes', "students' attitude", "students' attitudes"], ['student perception', 'student perceptions', 'students perception', 'students perceptions', "students' perception", "students' perceptions"], ['teacher attitude', 'teacher attitudes', 

In [4]:
# ============================================================
# EMBEDDINGS OF THE UNION VOCABULARY
# ============================================================
model = C.load_embedder(threads=THREADS)
# The batch progress bar of sentence-transformers does not render in an executed notebook.
with contextlib.redirect_stderr(io.StringIO()):
    emb = C.embed_unique(model, set(vocab_llm) | set(vocab_sc), batch_size=256)
T.mark("embeddings")
print(f"embedded {len(emb)} unique terms in {T.marks['embeddings'] - T.marks['vocabulary']:.0f} s")

embedded 121645 unique terms in 1011 s


In [5]:
# ============================================================
# ARMS: GLOBAL GRAPH, BACKBONE, LOUVAIN
# ============================================================
arms, graphs, backbones, parts = [], {}, {}, {}
for name, docs, tau in (("llm_k5", llm_docs, C.TAU_EDGE), ("scopus_all", sc_all, C.TAU_EDGE),
                        ("scopus_first5", sc_first5, C.TAU_EDGE), ("scopus_all_coword", sc_all, -1.0)):
    s, G, H, part = build_arm(name, docs, emb, tau, T)
    arms.append(s); graphs[name] = G; backbones[name] = H; parts[name] = part
    print(f"  {name}: nodes={s['global_nodes']} edges={s['global_edges']} isolated={s['global_isolated_nodes']} "
          f"bb={s['backbone_nodes']}/{s['backbone_edges']} Q={s['backbone_modularity_seed42']:.4f} "
          f"comm={s['backbone_communities_seed42']} hub={s.get('backbone_hub')} ({s.get('backbone_hub_share_of_edges', 0):.3f})", flush=True)
    wdeg = dict(H.degree(weight="weight"))
    pd.DataFrame({"keyword": list(H.nodes()), "community": [part.get(k) for k in H.nodes()],
                  "degree": [H.degree(k) for k in H.nodes()], "weighted_degree": [wdeg[k] for k in H.nodes()]}
                 ).sort_values("weighted_degree", ascending=False).to_csv(OUT / f"backbone_nodes_{name}.csv", index=False)
    pd.Series(part).value_counts().rename_axis("community").reset_index(name="n_keywords").to_csv(
        OUT / f"community_sizes_{name}.csv", index=False)
    pd.DataFrame([(u, v, d["weight"], d.get("sim_mean")) for u, v, d in H.edges(data=True)],
                 columns=["u", "v", "weight", "sim_mean"]).sort_values("weight", ascending=False).head(60).to_csv(
        OUT / f"backbone_top_edges_{name}.csv", index=False)
table = pd.DataFrame(arms)
table.to_csv(OUT / "arms_comparison.csv", index=False)

print("\nARMS COMPARISON")
print(table.set_index("arm").T.to_string())

  llm_k5: nodes=56635 edges=109022 isolated=18017 bb=408/608 Q=0.3560 comm=7 hub=mathematics education (0.515)


  scopus_all: nodes=97738 edges=541841 isolated=14337 bb=1337/5261 Q=0.4018 comm=46 hub=students (0.095)


  scopus_first5: nodes=65420 edges=116867 isolated=19333 bb=198/385 Q=0.4549 comm=17 hub=education (0.182)


  scopus_all_coword: nodes=97738 edges=2305259 isolated=48 bb=1760/12441 Q=0.3720 comm=12 hub=students (0.087)



ARMS COMPARISON
arm                                                   llm_k5 scopus_all scopus_first5 scopus_all_coword
tau                                                      0.4        0.4           0.4              -1.0
documents                                              52946      52946         52946             52946
keywords_total                                        264586     530670        250715            530670
keywords_per_document_mean                           4.99728  10.022853      4.735296         10.022853
keywords_per_document_median                             5.0        6.0           5.0               6.0
global_nodes                                           56635      97738         65420             97738
global_edges                                          109022     541841        116867           2305259
global_density                                      0.000068   0.000113      0.000055          0.000483
global_components                              

In [6]:
# ============================================================
# REPRODUCTION GATE (llm_k5 arm = published CRS)
# ============================================================
ref, g = C.CRS_REFERENCE, table[table.arm == "llm_k5"].iloc[0]
checks = {k: {"observed": int(g[o]), "reference": ref[k], "pass": int(g[o]) == ref[k]} for k, o in
          [("analyzed_documents", "documents"), ("nodes", "global_nodes"), ("edges", "global_edges"),
           ("components", "global_components"), ("lcc_nodes", "global_lcc_nodes"),
           ("backbone_nodes", "backbone_nodes"), ("backbone_edges", "backbone_edges"),
           ("backbone_communities", "backbone_communities_seed42")]}
checks["backbone_modularity"] = {"observed": float(g.backbone_modularity_seed42), "reference": ref["backbone_modularity"],
                                 "pass": abs(float(g.backbone_modularity_seed42) - ref["backbone_modularity"]) < 1e-6}
gate_pass = all(v["pass"] for v in checks.values())
print("REPRODUCTION GATE (llm_k5):", "PASS" if gate_pass else "FAIL")
for k, v in checks.items():
    print(f"  {k:22s} observed={v['observed']} reference={v['reference']} {'ok' if v['pass'] else 'MISMATCH'}")
assert gate_pass, "the llm_k5 arm does not reproduce the published CRS; the other arms are not interpretable"

REPRODUCTION GATE (llm_k5): PASS
  analyzed_documents     observed=52946 reference=52946 ok
  nodes                  observed=56635 reference=56635 ok
  edges                  observed=109022 reference=109022 ok
  components             observed=19856 reference=19856 ok
  lcc_nodes              observed=34316 reference=34316 ok
  backbone_nodes         observed=408 reference=408 ok
  backbone_edges         observed=608 reference=608 ok
  backbone_communities   observed=7 reference=7 ok
  backbone_modularity    observed=0.3560443171946043 reference=0.3560443171946043 ok


In [7]:
# ============================================================
# COMPARISONS BETWEEN ARMS (backbone concepts, partitions, documents, vocabulary)
# ============================================================
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

comp = {}
for a in ("scopus_all", "scopus_first5", "scopus_all_coword"):
    na, nl = set(backbones[a].nodes()), set(backbones["llm_k5"].nodes())
    # document-level agreement (documents with an unambiguous community in both arms)
    da = document_assignments(sc_all if a.startswith("scopus_all") else sc_first5, parts[a])
    dl = document_assignments(llm_docs, parts["llm_k5"])
    both = [(x, y) for x, y in zip(da, dl) if x not in (None, "tie") and y not in (None, "tie")]
    comp[f"llm_k5_vs_{a}"] = {
        "backbone_nodes_llm": len(nl), f"backbone_nodes_{a}": len(na), "shared_backbone_concepts": len(na & nl),
        "jaccard_backbone_concepts": len(na & nl) / max(1, len(na | nl)),
        "concept_partition_agreement_on_shared": C.partition_agreement(parts["llm_k5"], parts[a]),
        "documents_unambiguous_llm": int(sum(1 for y in dl if y not in (None, "tie"))),
        f"documents_unambiguous_{a}": int(sum(1 for x in da if x not in (None, "tie"))),
        "documents_compared": len(both),
        "document_nmi": float(normalized_mutual_info_score([y for _, y in both], [x for x, _ in both])) if len(both) > 1 else None,
        "document_ari": float(adjusted_rand_score([y for _, y in both], [x for x, _ in both])) if len(both) > 1 else None,
        "llm_backbone_concepts_absent_from_scopus_vocabulary": int(sum(1 for k in nl if k not in set(vocab_sc))),
    }
    pd.DataFrame(sorted(nl - na)).to_csv(OUT / f"llm_backbone_concepts_not_in_{a}_backbone.csv", index=False, header=["keyword"])
# per-document labels for both main arms (no text)
pd.DataFrame({"eid": eids, "community_llm_k5": document_assignments(llm_docs, parts["llm_k5"]),
              "community_scopus_all": document_assignments(sc_all, parts["scopus_all"]),
              "community_scopus_first5": document_assignments(sc_first5, parts["scopus_first5"])}).to_csv(
    OUT / "document_communities.csv", index=False)
# vocabulary overlap
overlap = {"llm_vocabulary": len(vocab_llm), "scopus_vocabulary": len(vocab_sc),
           "shared_terms": len(set(vocab_llm) & set(vocab_sc)),
           "share_llm_terms_present_in_scopus_vocabulary": len(set(vocab_llm) & set(vocab_sc)) / len(vocab_llm),
           "share_llm_keyword_instances_that_are_a_scopus_keyword_of_same_record": float(np.mean(
               [k in set(s) for d, s in zip(llm_docs, sc_all) for k in d]))}
T.mark("comparisons")

print("COMPARISONS WITH THE llm_k5 ARM")
for k, v in comp.items():
    print(k, {kk: (round(vv, 4) if isinstance(vv, float) else vv) for kk, vv in v.items() if kk != "concept_partition_agreement_on_shared"},
          "shared-concept NMI/ARI:", {kk: round(vv, 4) for kk, vv in v["concept_partition_agreement_on_shared"].items()})
print("\nVOCABULARY OVERLAP")
print(json.dumps(overlap, indent=2))

COMPARISONS WITH THE llm_k5 ARM
llm_k5_vs_scopus_all {'backbone_nodes_llm': 408, 'backbone_nodes_scopus_all': 1337, 'shared_backbone_concepts': 259, 'jaccard_backbone_concepts': 0.1743, 'documents_unambiguous_llm': 39477, 'documents_unambiguous_scopus_all': 41262, 'documents_compared': 30892, 'document_nmi': 0.0602, 'document_ari': 0.0216, 'llm_backbone_concepts_absent_from_scopus_vocabulary': 2} shared-concept NMI/ARI: {'shared_nodes': 259, 'nmi': 0.2346, 'ari': 0.1897}
llm_k5_vs_scopus_first5 {'backbone_nodes_llm': 408, 'backbone_nodes_scopus_first5': 198, 'shared_backbone_concepts': 102, 'jaccard_backbone_concepts': 0.2024, 'documents_unambiguous_llm': 39477, 'documents_unambiguous_scopus_first5': 31615, 'documents_compared': 24227, 'document_nmi': 0.0883, 'document_ari': 0.0618, 'llm_backbone_concepts_absent_from_scopus_vocabulary': 2} shared-concept NMI/ARI: {'shared_nodes': 102, 'nmi': 0.3808, 'ari': 0.3916}
llm_k5_vs_scopus_all_coword {'backbone_nodes_llm': 408, 'backbone_nodes_

In [8]:
# ============================================================
# SUMMARY, VALIDATION AND METADATA
# ============================================================
summary = {"documents": n, "records_without_scopus_keywords": n_no_scopus, "arms": arms,
           "vocabulary_fragmentation": frag, "english_residue": residue, "vocabulary_overlap": overlap,
           "comparisons": comp, "louvain_seeds": LOUVAIN_SEEDS}
C.write_json(summary, OUT / "summary.json")
C.write_json({"gate": "llm_k5 arm reproduces the published CRS (notebooks 2-5)", "pass": gate_pass, "checks": checks},
             OUT / "validation.json")
pd.DataFrame([{"arm": k, **{kk: vv for kk, vv in v.items() if kk != "largest_groups"}} for k, v in frag.items()]).to_csv(
    OUT / "vocabulary_fragmentation.csv", index=False)
meta.update({"completed_utc": C.now_utc(), "timings_seconds": T.marks, "status": "complete", "gate_pass": gate_pass})
C.write_json(meta, OUT / "metadata.json")

print("FOUR-ARM COMPARISON")
print(table[["arm", "documents", "keywords_per_document_mean", "global_nodes", "global_edges", "global_isolated_nodes",
             "global_lcc_fraction", "backbone_nodes", "backbone_edges", "backbone_communities_seed42",
             "backbone_modularity_seed42", "backbone_hub", "backbone_hub_share_of_edges",
             "share_documents_with_backbone_concept"]].round(4).to_string(index=False))
print("\ntimings (s):", T.marks)
print("written:", sorted(p.name for p in OUT.iterdir()))

FOUR-ARM COMPARISON
              arm  documents  keywords_per_document_mean  global_nodes  global_edges  global_isolated_nodes  global_lcc_fraction  backbone_nodes  backbone_edges  backbone_communities_seed42  backbone_modularity_seed42          backbone_hub  backbone_hub_share_of_edges  share_documents_with_backbone_concept
           llm_k5      52946                      4.9973         56635        109022                  18017               0.6059             408             608                            7                      0.3560 mathematics education                       0.5148                                 0.9159
       scopus_all      52946                     10.0229         97738        541841                  14337               0.8004            1337            5261                           46                      0.4018              students                       0.0954                                 0.9013
    scopus_first5      52946                      4.7353

## Check against the archived results

Not in the manuscript; the run is compared with the results archived before it.

In [9]:
# ============================================================
# CHECK AGAINST THE ARCHIVED RESULTS (not reported in the manuscript)
# ============================================================
print("This experiment is an additional analysis; none of its numbers appears in main.tex.")
rows = []
if "arms_comparison.csv" not in archived or "summary.json" not in archived:
    print("no archived results were found before this run; nothing to compare")
else:
    old = archived["arms_comparison.csv"].set_index("arm")
    new = table.set_index("arm")
    metrics = ["documents", "keywords_total", "keywords_per_document_mean", "global_nodes", "global_edges",
               "global_components", "global_lcc_nodes", "global_isolated_nodes", "backbone_nodes", "backbone_edges",
               "backbone_components", "backbone_lcc_nodes", "backbone_modularity_seed42", "backbone_modularity_mean",
               "backbone_communities_seed42", "backbone_communities_mean", "backbone_hub", "backbone_hub_degree",
               "backbone_hub_share_of_edges", "share_documents_with_backbone_concept"]
    for arm in new.index:
        for m in metrics:
            a, b = old.loc[arm, m], new.loc[arm, m]
            if isinstance(b, str) or isinstance(a, str):
                ok = str(a) == str(b)
            elif float(b).is_integer() and float(a).is_integer():
                ok = int(a) == int(b)
            else:
                ok = round(float(a), 6) == round(float(b), 6)
            rows.append({"arm": arm, "quantity": m, "archived": a, "this run": b, "flag": "match" if ok else "differs"})
    oc, nc = archived["summary.json"]["comparisons"], comp
    for k in nc:
        for m in ("shared_backbone_concepts", "documents_compared", "document_nmi", "document_ari"):
            a, b = oc[k][m], nc[k][m]
            ok = (a == b) if isinstance(b, int) else round(a, 6) == round(b, 6)
            rows.append({"arm": k, "quantity": m, "archived": a, "this run": b, "flag": "match" if ok else "differs"})
        for m in ("nmi", "ari"):
            a = oc[k]["concept_partition_agreement_on_shared"][m]; b = nc[k]["concept_partition_agreement_on_shared"][m]
            rows.append({"arm": k, "quantity": f"shared-concept {m}", "archived": a, "this run": b,
                         "flag": "match" if round(a, 6) == round(b, 6) else "differs"})
    oo = archived["summary.json"]["vocabulary_overlap"]
    for m, b in overlap.items():
        a = oo[m]
        ok = (a == b) if isinstance(b, int) else round(a, 6) == round(b, 6)
        rows.append({"arm": "vocabulary", "quantity": m, "archived": a, "this run": b, "flag": "match" if ok else "differs"})
    rows.append({"arm": "gate", "quantity": "llm_k5 reproduces the published CRS",
                 "archived": archived["validation.json"]["pass"], "this run": gate_pass,
                 "flag": "match" if archived["validation.json"]["pass"] == gate_pass else "differs"})
    check = pd.DataFrame(rows)
    pd.set_option("display.width", 250)
    print(check.to_string(index=False))
    n_diff = int((check.flag == "differs").sum())
    print(f"\n{len(check)} comparisons: {len(check) - n_diff} match, {n_diff} differ")

This experiment is an additional analysis; none of its numbers appears in main.tex.
                        arm                                                             quantity              archived              this run  flag
                     llm_k5                                                            documents                 52946                 52946 match
                     llm_k5                                                       keywords_total                264586                264586 match
                     llm_k5                                           keywords_per_document_mean               4.99728               4.99728 match
                     llm_k5                                                         global_nodes                 56635                 56635 match
                     llm_k5                                                         global_edges                109022                109022 match
                     llm_k5       